# NexusTrade — Portfolio Allocation Calculator
## Track and Optimize Position Sizing Across Portfolio

Visualize concentration risk en zorg dat je gediversifieerd blijft.

In [ ]:
import pandas as pd
import numpy as np

def calculate_allocation(portfolio_value, positions):
    """
    Calculate portfolio allocation
    
    Args:
        portfolio_value: Total portfolio value
        positions: List of dicts with 'ticker', 'shares', 'price', 'type'
                   type: 'etf', 'quality', 'growth', 'speculative', 'cash'
    
    Returns:
        DataFrame with allocation details
    """
    if not positions:
        return {"error": "No positions provided"}
    
    df = pd.DataFrame(positions)
    df['value'] = df['shares'] * df['price']
    df['allocation_pct'] = (df['value'] / portfolio_value) * 100
    
    # Status indicator
    def get_status(pct, position_type):
        if position_type == 'cash':
            return '✅' if pct >= 10 else '⚠️'
        elif position_type == 'speculative':
            if pct <= 3:
                return '✅'
            elif pct <= 10:
                return '⚠️'
            else:
                return '🔴'
        else:  # etf, quality, growth
            if pct <= 10:
                return '✅'
            elif pct <= 20:
                return '⚠️'
            else:
                return '🔴'
    
    df['status'] = df.apply(lambda row: get_status(row['allocation_pct'], row['type']), axis=1)
    
    # Sort by allocation descending
    df = df.sort_values('allocation_pct', ascending=False)
    
    # Summary by type
    type_summary = df.groupby('type')['value'].sum()
    type_pct = (type_summary / portfolio_value) * 100
    
    return {
        'positions': df,
        'type_summary': pd.DataFrame({
            'value': type_summary,
            'allocation_pct': type_pct
        }).sort_values('allocation_pct', ascending=False)
    }

## Example: Current Portfolio (60% DVLT Overexposure)

In [ ]:
# VOORBEELD: Huidige portfolio met DVLT overexposure
portfolio_value = 2850

current_positions = [
    {'ticker': 'DVLT', 'shares': 2055, 'price': 0.7386, 'type': 'speculative'},
    {'ticker': 'CASH', 'shares': 1, 'price': 341, 'type': 'cash'},
]

result = calculate_allocation(portfolio_value, current_positions)

print("═══════════════════════════════════════════")
print("CURRENT PORTFOLIO ALLOCATION")
print("═══════════════════════════════════════════")
print("\nPOSITIONS:")
print(result['positions'][['status', 'ticker', 'shares', 'price', 'value', 'allocation_pct', 'type']].to_string(index=False))
print("\n" + "─" * 47)
print(f"Total Portfolio: ${portfolio_value:,.2f}")
print("═══════════════════════════════════════════")

print("\nALLOCATION BY TYPE:")
print(result['type_summary'].to_string())

print("\n⚠️  ALLOCATION RULES:")
print("   ✅ Cash: ≥10%")
print("   ✅ Speculative: ≤3% (⚠️ ≤10%, 🔴 >10%)")
print("   ✅ Quality/Growth: ≤10% (⚠️ ≤20%, 🔴 >20%)")
print("   ✅ ETF: ≤10% per ETF")

# Critical warnings
dvlt_pct = result['positions'][result['positions']['ticker'] == 'DVLT']['allocation_pct'].values[0]
if dvlt_pct > 50:
    print("\n🚨 CRITICAL: DVLT position >50% — EMERGENCY RISK!")
    print("   This is not investing, it's gambling on one bet.")
    print("   EXIT PLAN: Sell 50% immediately, set stops on rest.")

## Target Portfolio: Balanced Allocation

In [ ]:
# TARGET: Balanced portfolio na DVLT exit
target_positions = [
    {'ticker': 'SPY', 'shares': 5, 'price': 600, 'type': 'etf'},      # 30% — Large cap ETF
    {'ticker': 'QQQ', 'shares': 2, 'price': 450, 'type': 'etf'},      # 9% — Tech ETF
    {'ticker': 'VOO', 'shares': 2, 'price': 450, 'type': 'etf'},      # 9% — S&P 500 ETF
    {'ticker': 'AAPL', 'shares': 10, 'price': 175, 'type': 'quality'},# 6% — Quality stock
    {'ticker': 'MSFT', 'shares': 4, 'price': 420, 'type': 'quality'}, # 6% — Quality stock
    {'ticker': 'NVDA', 'shares': 10, 'price': 90, 'type': 'growth'},  # 3% — Growth stock
    {'ticker': 'DVLT', 'shares': 500, 'price': 0.85, 'type': 'speculative'}, # 1.5% — Speculative (reduced)
    {'ticker': 'CASH', 'shares': 1, 'price': 585, 'type': 'cash'},    # ~20% — Cash reserve
]

target_result = calculate_allocation(portfolio_value, target_positions)

print("\n═══════════════════════════════════════════")
print("TARGET PORTFOLIO ALLOCATION (BALANCED)")
print("═══════════════════════════════════════════")
print("\nPOSITIONS:")
print(target_result['positions'][['status', 'ticker', 'shares', 'price', 'value', 'allocation_pct', 'type']].to_string(index=False))
print("\n" + "─" * 47)
print(f"Total Portfolio: ${portfolio_value:,.2f}")
print("═══════════════════════════════════════════")

print("\nALLOCATION BY TYPE:")
print(target_result['type_summary'].to_string())

print("\n✅ This is a healthy, diversified portfolio.")

## Portfolio Health Score

In [ ]:
def calculate_health_score(result, portfolio_value):
    """Calculate portfolio health score (0-100)"""
    score = 100
    issues = []
    
    df = result['positions']
    
    # Check cash position
    cash_pct = df[df['type'] == 'cash']['allocation_pct'].sum()
    if cash_pct < 5:
        score -= 20
        issues.append("🔴 Cash <5% — no dry powder for opportunities")
    elif cash_pct < 10:
        score -= 10
        issues.append("⚠️  Cash <10% — low buffer for emergencies")
    
    # Check concentration risk
    max_position = df['allocation_pct'].max()
    if max_position > 50:
        score -= 50
        issues.append(f"🔴 Largest position {max_position:.1f}% — EXTREME concentration")
    elif max_position > 20:
        score -= 30
        issues.append(f"🔴 Largest position {max_position:.1f}% — too concentrated")
    elif max_position > 10:
        score -= 10
        issues.append(f"⚠️  Largest position {max_position:.1f}% — monitor closely")
    
    # Check speculative exposure
    spec_pct = df[df['type'] == 'speculative']['allocation_pct'].sum()
    if spec_pct > 20:
        score -= 30
        issues.append(f"🔴 Speculative {spec_pct:.1f}% — gambling territory")
    elif spec_pct > 10:
        score -= 20
        issues.append(f"🔴 Speculative {spec_pct:.1f}% — too much risk")
    elif spec_pct > 3:
        score -= 10
        issues.append(f"⚠️  Speculative {spec_pct:.1f}% — above 3% target")
    
    # Check diversification
    num_positions = len(df[df['type'] != 'cash'])
    if num_positions == 1:
        score -= 30
        issues.append("🔴 Only 1 position — zero diversification")
    elif num_positions < 3:
        score -= 20
        issues.append("🔴 <3 positions — poor diversification")
    elif num_positions < 5:
        score -= 10
        issues.append("⚠️  <5 positions — could diversify more")
    
    score = max(0, score)  # Floor at 0
    
    return {
        'score': score,
        'grade': 'A' if score >= 90 else 'B' if score >= 80 else 'C' if score >= 70 else 'D' if score >= 60 else 'F',
        'issues': issues
    }

# Score current portfolio
current_health = calculate_health_score(result, portfolio_value)
print("\n═══════════════════════════════════════════")
print("CURRENT PORTFOLIO HEALTH SCORE")
print("═══════════════════════════════════════════")
print(f"Score: {current_health['score']}/100 (Grade: {current_health['grade']})")
if current_health['issues']:
    print("\nIssues:")
    for issue in current_health['issues']:
        print(f"   {issue}")
else:
    print("\n✅ No issues detected — portfolio is healthy!")

# Score target portfolio
target_health = calculate_health_score(target_result, portfolio_value)
print("\n═══════════════════════════════════════════")
print("TARGET PORTFOLIO HEALTH SCORE")
print("═══════════════════════════════════════════")
print(f"Score: {target_health['score']}/100 (Grade: {target_health['grade']})")
if target_health['issues']:
    print("\nIssues:")
    for issue in target_health['issues']:
        print(f"   {issue}")
else:
    print("\n✅ No issues detected — portfolio is healthy!")

## Interactive: Your Portfolio

In [ ]:
# ==== ENTER YOUR ACTUAL POSITIONS ====
MY_PORTFOLIO_VALUE = 2850

my_positions = [
    {'ticker': 'DVLT', 'shares': 2055, 'price': 0.7386, 'type': 'speculative'},
    {'ticker': 'CASH', 'shares': 1, 'price': 341, 'type': 'cash'},
    # Add more positions here
]
# ======================================

my_result = calculate_allocation(MY_PORTFOLIO_VALUE, my_positions)
my_health = calculate_health_score(my_result, MY_PORTFOLIO_VALUE)

print("\n🎯 YOUR PORTFOLIO:")
print(my_result['positions'][['status', 'ticker', 'allocation_pct', 'type']].to_string(index=False))
print(f"\nHealth Score: {my_health['score']}/100 (Grade: {my_health['grade']})")

if my_health['issues']:
    print("\n⚠️  Issues to Address:")
    for issue in my_health['issues']:
        print(f"   {issue}")

## Rebalancing Scenarios

In [ ]:
# Test verschillende rebalancing scenarios
scenarios = {
    'Current (60% DVLT)': current_positions,
    'Target (Balanced)': target_positions,
    'Conservative (80% ETF)': [
        {'ticker': 'SPY', 'shares': 6, 'price': 600, 'type': 'etf'},
        {'ticker': 'QQQ', 'shares': 3, 'price': 450, 'type': 'etf'},
        {'ticker': 'CASH', 'shares': 1, 'price': 570, 'type': 'cash'},
    ],
    'Aggressive (30% Growth)': [
        {'ticker': 'SPY', 'shares': 4, 'price': 600, 'type': 'etf'},
        {'ticker': 'NVDA', 'shares': 10, 'price': 90, 'type': 'growth'},
        {'ticker': 'TSLA', 'shares': 5, 'price': 180, 'type': 'growth'},
        {'ticker': 'DVLT', 'shares': 1000, 'price': 0.85, 'type': 'speculative'},
        {'ticker': 'CASH', 'shares': 1, 'price': 285, 'type': 'cash'},
    ]
}

comparison = []

for name, positions in scenarios.items():
    res = calculate_allocation(portfolio_value, positions)
    health = calculate_health_score(res, portfolio_value)
    
    type_sum = res['type_summary']
    
    comparison.append({
        'Scenario': name,
        'Score': health['score'],
        'Grade': health['grade'],
        'Cash %': type_sum.loc['cash', 'allocation_pct'] if 'cash' in type_sum.index else 0,
        'ETF %': type_sum.loc['etf', 'allocation_pct'] if 'etf' in type_sum.index else 0,
        'Spec %': type_sum.loc['speculative', 'allocation_pct'] if 'speculative' in type_sum.index else 0,
        'Quality %': type_sum.loc['quality', 'allocation_pct'] if 'quality' in type_sum.index else 0,
        'Growth %': type_sum.loc['growth', 'allocation_pct'] if 'growth' in type_sum.index else 0,
    })

df_comparison = pd.DataFrame(comparison)
print("\n═══════════════════════════════════════════")
print("PORTFOLIO SCENARIO COMPARISON")
print("═══════════════════════════════════════════")
print(df_comparison.to_string(index=False))
print("\n💡 Key Insight:")
print("Target (Balanced) provides best risk/reward with 90+ health score.")
print("Current portfolio gets F grade due to extreme concentration risk.")